In [5]:
#From the DE+IC tutorial 
import sqlite3
import pandas as pd

def create_connection(db_file):
    """ create a database connection to the SQLite database specified by the db_file 
    :param db_file: database file
    :return: Connection object or None
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
    except Error as e:
        print(e)
    return conn

#creating the .db
conn = create_connection("worldcup.db")

#reading the .sql
with open("worldcup.sql", "r", encoding="utf-8") as file:
    sql_script = file.read()

#executing the .sql, filling the .db
conn.executescript(sql_script)
conn.commit()

In [6]:
#From the DE+IC tutorial     
def run_query(conn, query):
    """
    Query all rows in the teams table
    :param conn: the Connection object
    :return: the results of executing the query
    """
     # Create a cursor 
    cur = conn.cursor()    
    # Send the query to the database
    cur.execute(query)
    # Extract the results of the query
    results = cur.fetchall()
    # Return the results 
    return results

#Query test for all table headers
tables = run_query(conn, 'SELECT name FROM sqlite_schema WHERE type="table"')
print(tables)

[('Country',), ('Stadium',), ('Team',), ('Player',), ('Game',), ('Goal',), ('Assist',), ('Penalty',), ('Card',), ('Substitution',)]


In [7]:
#From the DE+IC tutorial 
def convert_db_table_to_DF(conn, table):
    # get the names of the attributes in the database table
    header_query = "SELECT name FROM pragma_table_info('" + table + "') ORDER BY cid;"
    cols_init = run_query(conn, header_query)
    cols = [cols_init[i][0] for i in range(len(cols_init))]
    # get the records of the table
    content_query = "Select * FROM " + table
    data = run_query(conn, content_query)
    df = pd.DataFrame(data, columns = cols)
    return df 
df_sim = convert_db_table_to_DF(conn, "Player")
print(f"Player table shape: {df_sim.shape}")
print(df_sim.head(15))

Player table shape: (57, 5)
    id               name  team_id    position               club
0    1    Julian Quinones        1     Forward       Club America
1    2       Raul Jimenez        1     Forward             Fulham
2    3   Roberto Alvarado        1  Midfielder        Guadalajara
3    4          Erik Lira        1  Midfielder          Cruz Azul
4    5       Cesar Montes        1    Defender   Lokomotiv Moscow
5    6    Bryan Gutierrez        1  Midfielder             Chivas
6    7     Jesus Gallardo        1     Forward             Toluca
7    8      Gilberto Mora        1  Midfielder            Tijuana
8    9   Santiago Gimenez        1     Forward           AC Milan
9   10      Edson Alvarez        1    Defender           West Ham
10  11    Ronwen Williams        2      Keeper  Mamelodi Sundowns
11  12  Sphephelo Sithole        2    Defender            Tondela
12  13     Teboho Mokoena        2  Midfielder  Mamelodi Sundowns
13  14        Lyle Foster        2     Forward  

In [ ]:
def hash_string(s, m):
    """
    Map a string to an index in [0, m-1].
    Polynomial rolling hash base 31 applying %m after every step,
    reducing h back down inside the [0, m-1] range to avoid large values.
    """
    h = 0
    for ch in s:
        h = (h * 31 + ord(ch)) % m
    return h

hash_values = []
m = len(df_sim["name"])
for name in df_sim["name"]:
    hash_values.append(hash_string(name, m))
df_sim["hash_index"] = hash_values

print(df_sim[["name", "hash_index", "club"]])

In [ ]:
counts = df_sim["hash_index"].value_counts() #counts hash frequencies
colliding_buckets = counts[counts > 1] #If hash occurs >1, it collides

n_collisions = (colliding_buckets - 1).sum()
collision_rate = n_collisions / m 

print(f"m = {m}")
print(f"Colliding buckets: {len(colliding_buckets)}")
print(f"Total collisions: {n_collisions}")
print(f"Collision rate: {collision_rate:.2%}")

In [ ]:
#The buckets with collissions
for idx, group in df_sim.groupby("hash_index"):
    if len(group) > 1:
        print(idx, list(group["name"]))

In [ ]:
#The inserted high-similarity dummy players land in completely different buckets than their original, 
#the hash function is insensitive to their similarity
targets = ["Erik Lira", "Erick Lira", "Lee Kang-in", "Lee Kang In", "Lionel Messi", "Leonel Messi"]
print(df_sim[df_sim["name"].isin(targets)][["name", "hash_index"]])